# 8.3. Network in Network (NiN)
D2L의 Network in Network (NiN)장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 기존 CNN의 문제점

LeNet, AlexNet, VGG는 기본적으로 비슷한 구조를 사용한다.

```text
Conv + Pooling
↓
Conv + Pooling
↓
...
↓
Flatten
↓
Fully Connected
↓
Fully Connected
↓
Output
```

앞부분에서 Convolution을 이용해 특징을 추출하고, 마지막엔 Fully Connected Layer를 이용해 분류한다. 여기엔 문제가 있다.

VGG처럼 이미지 크기와 채널 수가 큰 상태에서 Fully Connected Layer를 사용하면 파라미터 수가 매우 많아진다. 

NiN(Nutwork in Network)은 이 문제를 해결하기 위해

1. 1x1 Convolution
2. Global Average Pooling

이라는 아이디어를 사용한다.

## 2. NiN의 핵심 아이디어

NiN은 기존 CNN에서 두 가지를 변경한다.

기존 CNN
```text
일반 Conv
↓
일반 Conv
↓
Pooling
↓
...
↓
Flatten
↓
Fully Connected
```

NiN
```text
일반 Conv
↓
1×1 Conv
↓
1×1 Conv
↓
Pooling
↓
...
↓
Global Average Pooling
```

Conv Block안에 1x1 Conv를 추가하고, 마지막의 거대한 Fully Connected Layer를 제거하고 Global Average Pooling으로 바로 분류한다.

NiN의 목적은 단순히 모델을 작게 만드는게 아니고 각 위치에서 채널 정보를 더 복잡하게 조합하는 것도 중요한 목적이다.

## 3. 1x1 Convolution이란?

### 1x1 Convolution

1x1 Conv는 조금 이상해보인다. 일반적인 3x3 Conv는 주변 픽셀까지 같이 보는데 1x1는 한 위치만 본다.

1x1 Conv는 주변 픽셀을 보는게 아니라 같은 위치에 존재하는 여러 채널의 값을 섞는다.

[R, G, B] -> 가중합 -> 새로운 특징 을 만드는 느낌이다.

## 4. 1x1 Conv는 채널 방향 Fully Connected Layer

여기가 중요하다.

### 1x1 Conv의 진짜 의미

만약 입력이 이렇다고 하자 [batch, 64, H, W]

각 위치마다 64개의 특징값이 존재한다. 어느 한 위치 (h, w)만 본다면

[x1, x2, x3, ..., x64] 같은 64차원 벡터가 있는 것과 같다.

여기에 만약에

```py
nn.Conv2d(
    in_channels=64,
    out_channels=32,
    kernel_size=1
)
```

이걸 적용하면 각 위치마다 64개 특징 -> 32개 특징으로 변환한다.

결과는 [batch, 32, H, W]가 된다.

1x1 Conv는 이미지의 각 위치마다 동일한 Fully Connected Layer를 적용하는 것이라고 생각하면 된다.

## 5. 왜 굳이 1x1 Conv를 사용할까?

### 1. 채널을 조합할 수 있다.

예를 들어 어떤 위치에 아래 같은 특징들이 여러 채널에 존재한다.

```text
edge
color
texture
curve
...
```

1x1 Conv는 이런 채널들을 가중합해서 새로운 더 복잡한 특징을 만들 수 있다.

### 2. 채널을 바꿀 수 있다.

예를 들어 

    256 channels -> 1x1 Conv -> 64 channels

처럼 줄이거나 반대로 늘릴수 있다.

    64 channels -> 1x1 Conv -> 256 channels

### 3. 비선형성을 추가할 수 있다.

    1x1 Conv -> ReLU -> 1x1 Conv -> ReLU

를 사용하면 공간 크기를 바꾸지 않고도 신경망을 더 깊고 복잡하게 만들 수 있다.

## 6. NiN Block

NiN은 다음 구조를 하나의 Block으로 사용한다.

```text
일반 Conv
↓
ReLU
↓
1×1 Conv
↓
ReLU
↓
1×1 Conv
↓
ReLU
```

첫 번째 Conv는 주변 공간에서 특징 추출을 담당한다. 그 다음 1x1 Conv들은 각 위치에서 채널 특징을 조합한다. 

NiN Block은 `공간 특징 추출 + 채널 특징 조합`이라고 생각할 수 있다.

In [2]:
def nin_block(out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.LazyConv2d(
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        ),
        nn.ReLU(),

        nn.LazyConv2d(
            out_channels,
            kernel_size=1
        ),
        nn.ReLU(),

        nn.LazyConv2d(
            out_channels,
            kernel_size=1
        ),
        nn.ReLU()
    )

## 7. VGG Block과 NiN Block 비교

### VGG Block

```text
3×3 Conv
↓
ReLU
↓
3×3 Conv
↓
ReLU
↓
Pooling
```

VGG는 공간적인 특징을 계속 추출한다.

### NiN Block

```text
일반 Conv
↓
ReLU
↓
1×1 Conv
↓
ReLU
↓
1×1 Conv
↓
ReLU
↓
Pooling
```

첫 Conv에서 공간적인 특징을 찾고 1x1 Conv에서 같은 위치에 존재하는 채널 특징들을 조합한다.

    VGG: 공간 특징 → 공간 특징 → 공간 특징
    NiN: 공간 특징 → 채널 조합 → 채널 조합

## 8. NiN 전체 구조

NiN은 AlexNet과 비슷한 크기의 Conv kernel을 사용한다. 전체적인 구조는 이렇다.

```text
Input

↓ NiN Block
11×11 Conv
1×1 Conv
1×1 Conv

↓ MaxPool

↓ NiN Block
5×5 Conv
1×1 Conv
1×1 Conv

↓ MaxPool

↓ NiN Block
3×3 Conv
1×1 Conv
1×1 Conv

↓ MaxPool

↓ Dropout

↓ NiN Block
3×3 Conv
1×1 Conv
1×1 Conv

↓ Global Average Pooling

↓ Output
```

마지막에 Fully Connected Layer가 없다. 이것이 큰 차이이다.

## 9. Global Average Pooling

NiN은 마지막 FCL을 없앴다. 대신 마지막 feature map의 채널 수를 클래스 개수와 같게 만든다.

예를 들어서 Fashion-MNIST는 클래스가 10개다.

그럼 마지막 출력은 [batch, 10, 5, 5] 처럼 만든다.

여기서 채널 하나가 클래스 하나에 대응한다고 생각할 수 있는데 예를 들어서

```text
Channel 0 → T-shirt
Channel 1 → Trouser
Channel 2 → Pullover
...
```

각 채널에는 5x5 = 25개의 값이 있다. Global Average Pooling은 25개의 값을 모두 평균 낸다.

```text
[batch, 10, 5, 5]

↓ Global Average Pooling

[batch, 10, 1, 1]

↓ Flatten

[batch, 10]
```

결과적으로 클래스마다 하나의 logit이 만들어진다.

## 10. Fully Connected가 왜 필요 없어질까?

기존 CNN에서는 

```text
Feature Map
↓
Flatten
↓
Fully Connected
↓
10 Classes
```

처럼 분류한다. NiN에서는 마지막 Conv 출력 채널 자체를 클래스 개수와 같게 만든다. 그리고 각 채널 전체의 평균을 구한다.

예를 들어 고양이 분류 모델이라고 생각하면,

```text
고양이 채널의 여러 위치

0.1 0.2 0.8
0.4 0.9 0.7
0.1 0.3 0.6
```

전체 평균을 이용해서 고양이라는 특징이 이미지 전체적으로 얼마나 강하게 나타났는지?를 하나의 값으로 만든다고 보면 된다.

그래서 큰 FCL이 없어도 분류가 가능하다. 이 때문에 NiN은 파라미터 수를 크게 줄일 수 있었다.

## 11. NiN 구현과 출력크기 확인

In [3]:
class NiN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.net = nn.Sequential(

            nin_block(
                96,
                kernel_size=11,
                stride=4,
                padding=0
            ),

            nn.MaxPool2d(
                kernel_size=3,
                stride=2
            ),

            nin_block(
                256,
                kernel_size=5,
                stride=1,
                padding=2
            ),

            nn.MaxPool2d(
                kernel_size=3,
                stride=2
            ),

            nin_block(
                384,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.MaxPool2d(
                kernel_size=3,
                stride=2
            ),

            nn.Dropout(0.5),

            # 마지막 채널 수 = 클래스 수
            nin_block(
                num_classes,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            # 각 채널의 H, W 전체 평균
            nn.AdaptiveAvgPool2d((1, 1)),

            nn.Flatten()
        )

    def forward(self, x):
        return self.net(x)

In [4]:
model = NiN(num_classes=10)

X = torch.randn(1, 1, 224, 224)

for layer in model.net:
    X = layer(X)

    print(
        layer.__class__.__name__,
        X.shape
    )

Sequential torch.Size([1, 96, 54, 54])
MaxPool2d torch.Size([1, 96, 26, 26])
Sequential torch.Size([1, 256, 26, 26])
MaxPool2d torch.Size([1, 256, 12, 12])
Sequential torch.Size([1, 384, 12, 12])
MaxPool2d torch.Size([1, 384, 5, 5])
Dropout torch.Size([1, 384, 5, 5])
Sequential torch.Size([1, 10, 5, 5])
AdaptiveAvgPool2d torch.Size([1, 10, 1, 1])
Flatten torch.Size([1, 10])


## 12. 오늘의 정리

- NiN은 기존 CNN의 거대한 Fully Connected Layer 문제를 해결하려고 등장했다.
- NiN의 핵심은 `1×1 Conv`와 `Global Average Pooling`이다.
- `1×1 Conv`는 주변 픽셀을 보는 것이 아니라 같은 위치의 채널들을 조합한다.
- 따라서 `1×1 Conv`는 각 위치에 적용하는 작은 Fully Connected Layer처럼 생각할 수 있다.
- `1×1 Conv`를 사용하면 H, W를 유지하면서 채널 수를 변경할 수 있다.
- `1×1 Conv + ReLU`를 반복하면 공간 크기를 유지하면서 비선형성을 추가할 수 있다.
- NiN Block은 `일반 Conv → 1×1 Conv → 1×1 Conv` 구조이다.
- 마지막 feature map의 채널 수를 클래스 개수와 같게 만든다.
- Global Average Pooling은 각 채널의 모든 공간 값을 평균낸다.
- 따라서 거대한 Fully Connected Layer 없이 바로 분류할 수 있다.
- Fully Connected Layer를 제거하면서 파라미터 수를 크게 줄일 수 있다.
- `1×1 Conv`와 Global Average Pooling은 이후 많은 CNN 구조에 영향을 주었다.